# 01_07_build_kedr_obshiy_chod_drp

Снимок Kedr «Общий ЧОД» в озеро для витрины Jan–Jun.

1. ИНН из `final_df_period_*.csv` (или явный список).
2. **Preflight** по месяцам `202601`–`202606` для `v_detail_dmkb/dmsb/dkb` (`ok` / `empty` / `broken`).
3. Батч `sum(nbi_ssp)` только по `ok` источникам; `kedr_obshiy_chod = max` по доступным.
4. Load → `sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month`.
5. Витрина [`01_07_acq_dash_jan_jun_mpos.ipynb`](01_07_acq_dash_jan_jun_mpos.ipynb) читает эту таблицу (не live Kedr).

**Run All** после того, как period CSV уже собран.


In [ ]:
import math
import re
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Period Jan–Jun 2026
target_yearmms = [202601, 202602, 202603, 202604, 202605, 202606]
report_months = [f'{y // 100:04d}-{(y % 100):02d}' for y in target_yearmms]

# INN scope: period CSV from jan_jun notebook (override if needed)
final_df_csv_path = Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_06_mpos.csv')

target_table = 'sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month'
save_table_orc_name = 'shestopalov_kedr_obshiy_chod_inn_month_2026_01_2026_06.orc'

chunk_size = 800
mem_limit = '8g'
kedr_sources = {
    'dmkb': 'Kedr.v_detail_dmkb',
    'dmsb': 'Kedr.v_detail_dmsb',
    'dkb': 'Kedr.v_detail_dkb',
}

output_dir = Path('/home/jovyan/documents/Equaring/Data')
preflight_csv_path = output_dir / 'kedr_obshiy_chod_preflight_2026_01_2026_06.csv'
overlap_csv_path = output_dir / 'kedr_obshiy_chod_src_cnt_gt1_lake_2026_01_2026_06.csv'

print('target_table =', target_table)
print('yearmms =', target_yearmms)
print('final_df_csv_path =', final_df_csv_path, '| exists=', final_df_csv_path.exists())


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s


def in_sql_list(values):
    clean = [str(v).strip() for v in values if str(v).strip()]
    if not clean:
        return "''"
    return ', '.join(["'" + v.replace("'", "''") + "'" for v in clean])


def chunked(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

dl = connect(
    to='DATALAKE',
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
dl._init_connection()

print('Impala + Datalake initialized')


## 0) INN scope из final_df


In [ ]:
if not final_df_csv_path.exists():
    raise RuntimeError(
        f'Нет period CSV: {final_df_csv_path}. '
        'Сначала собери final_df_period в 01_07_acq_dash_jan_jun_mpos '
        '(можно с run_kedr_obshiy_chod_enrich=False).'
    )

final_src_df = pd.read_csv(final_df_csv_path, dtype=str, low_memory=False)
inn_col = 'inn' if 'inn' in final_src_df.columns else None
if inn_col is None:
    raise RuntimeError(f'Нет колонки inn в {final_df_csv_path}. cols={list(final_src_df.columns)[:30]}')

inn_values = sorted({x for x in final_src_df[inn_col].map(normalize_inn).dropna().tolist()})
print(f'final_df rows={len(final_src_df):,} | unique INN={len(inn_values):,}')
display(pd.DataFrame({'inn': inn_values}).head(10))


## 1) Preflight месяцев Kedr (`ok` / `empty` / `broken`)

Для каждого `yearmm` × источника: probe `LIMIT 1`, затем `count` (если probe ок).
Disk I/O / missing HDFS → `broken` (месяц×источник не грузим).


In [ ]:
# Preflight: month x source availability
preflight_rows = []

for src_name, fq_table in kedr_sources.items():
    for yyyymm in target_yearmms:
        status = 'ok'
        rows_cnt = None
        inns_cnt = None
        err = None
        # light probe
        sql_probe = f"select 1 as x from {fq_table} where yearmm = {yyyymm} limit 1"
        try:
            with imp:
                imp.execute(f'set MEM_LIMIT={mem_limit}')
                probe_df = imp.fetch(sql_probe)
            if probe_df is None or len(probe_df) == 0:
                status = 'empty'
            else:
                sql_cnt = f"""
                select
                  count(*) as rows_cnt,
                  count(distinct cast(inn as string)) as inns
                from {fq_table}
                where yearmm = {yyyymm}
                """
                with imp:
                    imp.execute(f'set MEM_LIMIT={mem_limit}')
                    cnt_df = imp.fetch(sql_cnt)
                if cnt_df is not None and len(cnt_df):
                    rows_cnt = int(pd.to_numeric(cnt_df.iloc[0]['rows_cnt'], errors='coerce') or 0)
                    inns_cnt = int(pd.to_numeric(cnt_df.iloc[0]['inns'], errors='coerce') or 0)
                    if rows_cnt <= 0:
                        status = 'empty'
                else:
                    status = 'empty'
        except Exception as exc:
            status = 'broken'
            err = f'{type(exc).__name__}: {str(exc)[:300]}'
            print(f'[preflight BROKEN] {src_name} yearmm={yyyymm}: {err}')

        preflight_rows.append({
            'src_name': src_name,
            'table_name': fq_table,
            'yearmm': yyyymm,
            'report_month': f'{yyyymm // 100:04d}-{(yyyymm % 100):02d}',
            'status': status,
            'rows_cnt': rows_cnt,
            'inns_cnt': inns_cnt,
            'error': err,
        })

preflight_df = pd.DataFrame(preflight_rows)
print('=== Kedr month preflight ===')
display(preflight_df)
preflight_df.to_csv(preflight_csv_path, index=False, encoding='utf-8-sig')
print('Saved:', preflight_csv_path)

ok_map = (
    preflight_df.loc[preflight_df['status'] == 'ok']
    .groupby('yearmm')['src_name']
    .apply(lambda s: sorted(set(s.tolist())))
    .to_dict()
)
print('OK sources by yearmm:')
for y, srcs in sorted(ok_map.items()):
    print(f'  {y}: {srcs}')

months_with_any_ok = sorted(ok_map.keys())
months_all_broken = [y for y in target_yearmms if y not in ok_map]
if months_all_broken:
    print('WARNING: no OK Kedr sources for yearmm =', months_all_broken)
if not months_with_any_ok:
    raise RuntimeError('Ни один месяц/источник Kedr не доступен (все broken/empty). Чини Kedr или сузь период.')


## 2) Батч выгрузка Общий ЧОД по ИНН × month (только `ok` источники)


In [ ]:
def build_kedr_sql_for_sources(inn_scope, yyyymm, src_names):
    """Build union SQL only for sources that passed preflight for this yearmm."""
    inn_sql = in_sql_list(inn_scope)
    parts_sql = []
    for src_name in src_names:
        fq = kedr_sources[src_name]
        parts_sql.append(f"""
    select cast(inn as string) as inn, sum(cast(nbi_ssp as double)) as chod_value, '{src_name}' as src_name
    from {fq}
    where yearmm = {yyyymm}
      and cast(inn as string) in ({inn_sql})
    group by cast(inn as string)
""")
    union_body = "\n    union all\n".join(parts_sql)
    return f"""
with src as (
{union_body}
)
select
    inn,
    max(case when src_name = 'dmkb' then chod_value end) as chod_dmkb,
    max(case when src_name = 'dmsb' then chod_value end) as chod_dmsb,
    max(case when src_name = 'dkb' then chod_value end) as chod_dkb,
    max(chod_value) as kedr_obshiy_chod,
    sum(case when src_name = 'dmkb' then 1 else 0 end)
      + sum(case when src_name = 'dmsb' then 1 else 0 end)
      + sum(case when src_name = 'dkb' then 1 else 0 end) as src_cnt
from src
group by inn
"""


parts = []
for yyyymm in months_with_any_ok:
    src_names = ok_map[yyyymm]
    total_chunks = max(1, math.ceil(len(inn_values) / chunk_size))
    print(f'yearmm={yyyymm}: sources={src_names}, inns={len(inn_values):,}, chunks={total_chunks}')
    for chunk_num, inn_scope in enumerate(chunked(inn_values, chunk_size), start=1):
        sql_text = build_kedr_sql_for_sources(inn_scope, int(yyyymm), src_names)
        try:
            with imp:
                imp.execute(f'set MEM_LIMIT={mem_limit}')
                part_df = imp.fetch(sql_text)
        except Exception as exc:
            print(f'  chunk {chunk_num}/{total_chunks} FAILED: {type(exc).__name__}: {str(exc)[:200]}')
            # mark this yearmm sources broken for remaining? continue other chunks carefully
            raise RuntimeError(
                f'Kedr fetch failed yearmm={yyyymm} chunk={chunk_num}. '
                f'Re-run preflight; likely partition went broken mid-load. {type(exc).__name__}'
            ) from exc

        if part_df is None or len(part_df) == 0:
            print(f'  chunk {chunk_num}/{total_chunks}: 0 rows')
            continue
        part_df = part_df.copy()
        part_df.columns = [str(c).strip().lower() for c in part_df.columns]
        part_df['inn'] = part_df['inn'].map(normalize_inn)
        part_df['yearmm'] = int(yyyymm)
        part_df['report_month'] = f'{int(yyyymm) // 100:04d}-{(int(yyyymm) % 100):02d}'
        part_df['ok_sources'] = ','.join(src_names)
        parts.append(part_df)
        print(f'  chunk {chunk_num}/{total_chunks}: {len(part_df):,} rows')

if parts:
    kedr_load_df = pd.concat(parts, ignore_index=True)
    kedr_load_df = (
        kedr_load_df.groupby(['inn', 'yearmm', 'report_month'], as_index=False)
        .agg(
            chod_dmkb=('chod_dmkb', 'max'),
            chod_dmsb=('chod_dmsb', 'max'),
            chod_dkb=('chod_dkb', 'max'),
            kedr_obshiy_chod=('kedr_obshiy_chod', 'max'),
            src_cnt=('src_cnt', 'max'),
            ok_sources=('ok_sources', 'first'),
        )
    )
else:
    kedr_load_df = pd.DataFrame(columns=[
        'inn', 'yearmm', 'report_month', 'chod_dmkb', 'chod_dmsb', 'chod_dkb',
        'kedr_obshiy_chod', 'src_cnt', 'ok_sources',
    ])

# ensure all INN x OK months present (left skeleton) — optional densify
skel = pd.MultiIndex.from_product(
    [inn_values, months_with_any_ok], names=['inn', 'yearmm']
).to_frame(index=False)
skel['report_month'] = skel['yearmm'].map(lambda y: f'{int(y) // 100:04d}-{(int(y) % 100):02d}')
kedr_load_df = skel.merge(kedr_load_df, on=['inn', 'yearmm', 'report_month'], how='left')

kedr_load_df['load_dt'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
kedr_load_df['source_file'] = 'Kedr.v_detail_*'

overlap_df = kedr_load_df.loc[pd.to_numeric(kedr_load_df['src_cnt'], errors='coerce').fillna(0) > 1].copy()
filled_pct_tmp = float(kedr_load_df['kedr_obshiy_chod'].notna().mean() * 100)
print(f'kedr_load_df rows={len(kedr_load_df):,} | filled={filled_pct_tmp:.2f}% | src_cnt>1={len(overlap_df):,}')
display(kedr_load_df.head(20))
if len(overlap_df):
    overlap_df.to_csv(overlap_csv_path, index=False, encoding='utf-8-sig')
    print('overlap csv:', overlap_csv_path)

cov = (
    kedr_load_df.assign(_filled=kedr_load_df['kedr_obshiy_chod'].notna().astype(int))
    .groupby('report_month', as_index=False)
    .agg(rows=('inn', 'size'), filled=('_filled', 'sum'), inns=('inn', 'nunique'))
)
cov['filled_pct'] = cov['filled'] / cov['rows'] * 100
print('Coverage by month:')
display(cov)


## 3) Prepare load + DROP/CREATE/ORC в озеро


In [ ]:
load_cols = [
    'inn', 'report_month', 'yearmm',
    'chod_dmkb', 'chod_dmsb', 'chod_dkb',
    'kedr_obshiy_chod', 'src_cnt', 'ok_sources',
    'load_dt', 'source_file',
]
final_load_df = kedr_load_df[load_cols].copy()

# dedupe key
dup = final_load_df.groupby(['inn', 'yearmm']).size().reset_index(name='cnt')
dup_cnt = int((dup['cnt'] > 1).sum())
print('duplicate keys (inn, yearmm) =', dup_cnt)
if dup_cnt:
    display(dup[dup['cnt'] > 1].head(20))
    raise RuntimeError('Дубли по (inn, yearmm)')

for c in ['chod_dmkb', 'chod_dmsb', 'chod_dkb', 'kedr_obshiy_chod', 'src_cnt', 'yearmm']:
    final_load_df[c] = pd.to_numeric(final_load_df[c], errors='coerce')

final_load_df = final_load_df.fillna({
    'inn': '',
    'report_month': '',
    'ok_sources': '',
    'load_dt': '',
    'source_file': 'Kedr.v_detail_*',
})

print('final_load_df rows =', len(final_load_df))
display(final_load_df.head(10))


In [ ]:
def load_to_datalake_from_file(local_file_path: str, table: str, dest_catalog: str = 'external', cleanup_before_copy: bool = True):
    schema = table.split('.')[0]
    table_name = table.split('.')[-1]
    hdfs_path = f'/warehouse/tablespace/{dest_catalog}/hive/{schema}.db/{table_name}'

    if cleanup_before_copy:
        subprocess.run(['hdfs', 'dfs', '-rm', '-r', '-f', f'{hdfs_path}/*'], check=False)

    subprocess.run(['hdfs', 'dfs', '-copyFromLocal', '-f', local_file_path, f'{hdfs_path}/{local_file_path}'], check=True)
    out = subprocess.run(['hdfs', 'dfs', '-ls', '-h', hdfs_path], capture_output=True, text=True, check=True)
    print(f'Files in HDFS path {hdfs_path}:\n{out.stdout}')


load_df = final_load_df.copy()
load_df.to_orc(save_table_orc_name, index=False)
print('ORC prepared:', save_table_orc_name, 'rows=', len(load_df))

create_sql = f"""
create external table if not exists {target_table} (
    inn string,
    report_month string,
    yearmm bigint,
    chod_dmkb double,
    chod_dmsb double,
    chod_dkb double,
    kedr_obshiy_chod double,
    src_cnt bigint,
    ok_sources string,
    load_dt string,
    source_file string
)
stored as orc
tblproperties ('transactional'='false')
"""

with dl:
    dl.execute(f'drop table if exists {target_table}')
    dl.execute(create_sql)

load_to_datalake_from_file(save_table_orc_name, target_table, dest_catalog='external', cleanup_before_copy=True)

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')

print('Reload completed:', target_table)


## 4) Post-load DQ


In [ ]:
sql_dq = f"""
select
  count(*) as rows_cnt,
  count(distinct cast(inn as string)) as inns,
  count(distinct cast(yearmm as bigint)) as months_cnt,
  sum(case when kedr_obshiy_chod is not null then 1 else 0 end) as filled_rows,
  sum(coalesce(kedr_obshiy_chod, 0.0)) as kedr_obshiy_chod_sum
from {target_table}
"""

sql_monthly = f"""
select
  report_month,
  cast(yearmm as bigint) as yearmm,
  count(*) as rows_cnt,
  sum(case when kedr_obshiy_chod is not null then 1 else 0 end) as filled_rows,
  sum(coalesce(kedr_obshiy_chod, 0.0)) as kedr_sum
from {target_table}
group by report_month, cast(yearmm as bigint)
order by yearmm
"""

sql_dup = f"""
select inn, cast(yearmm as bigint) as yearmm, count(*) as cnt
from {target_table}
group by inn, cast(yearmm as bigint)
having count(*) > 1
limit 20
"""

sql_sample = f"""
select *
from {target_table}
where kedr_obshiy_chod is not null
order by kedr_obshiy_chod desc
limit 30
"""

with imp:
    dq_df = imp.fetch(sql_dq)
    monthly_df = imp.fetch(sql_monthly)
    dup_df = imp.fetch(sql_dup)
    sample_df = imp.fetch(sql_sample)

print('DQ summary:')
display(dq_df)
print('By month:')
display(monthly_df)
print('Duplicates (should be empty):')
display(dup_df)
print('Top sample:')
display(sample_df)

print('Preflight reminder (broken months stay NULL in mart join):')
display(preflight_df.loc[preflight_df['status'] != 'ok'])
